# 04 · Kiểm định release và chuẩn bị bàn giao
**Câu hỏi:** release có đúng hợp đồng, có thể truy vết từng loại trừ, và cần mang gì sang máy trường?

**Đầu vào:** nguồn, metadata, waveform, records, thống kê train. **Đầu ra:** audit.json, dataset card và ba manifest accepted. **Tiếp theo:** 05 kiểm tra batch.

Ready chỉ có nghĩa đạt hợp đồng dữ liệu kỹ thuật. Nó không chứng nhận sạch nhiễu, nhãn đúng hoàn toàn, mô hình chính xác hay dùng lâm sàng. Bản notebook sửa 01/11/2026 chưa chạy toàn bộ; báo cáo 24/10/2026 là bằng chứng lịch sử của bản cũ.

In [ ]:
from pathlib import Path
import os
import sys

def tim_thu_muc_project():
    explicit = os.environ.get("PTBXL_PROJECT_ROOT")
    candidates = [Path(explicit).expanduser().resolve()] if explicit else [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "ptbxl_pipeline").is_dir() and (candidate / "requirements-ptbxl.txt").is_file():
            return candidate.resolve()
    raise FileNotFoundError("Không tìm thấy project. Đặt PTBXL_PROJECT_ROOT tới thư mục chứa ptbxl_pipeline và requirements-ptbxl.txt.")

ROOT = tim_thu_muc_project()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RAW_DIR = Path(os.environ.get("PTBXL_RAW_DIR", str(ROOT / "data/raw/ptbxl/1.0.3"))).expanduser().resolve()
CACHE_DIR = ROOT / "data/cache"
OUTPUT_DIR = Path(os.environ.get("PTBXL_OUTPUT_DIR", str(ROOT / "data/processed/ptbxl_v1.0.3_500hz_minimal"))).expanduser().resolve()
REPORT_DIR = ROOT / "artifacts/ptbxl_reports" / OUTPUT_DIR.name
TARGET_FS = 500
PREPROCESSING = "minimal"
EXPECTED_RECORDS = 21799
print("Project:", ROOT)
print("Dữ liệu gốc:", RAW_DIR)
print("Bộ xuất:", OUTPUT_DIR)

In [ ]:
from ptbxl_pipeline.core import PipelineConfig

config = PipelineConfig(
    raw_dir=RAW_DIR,
    output_dir=OUTPUT_DIR,
    target_fs=TARGET_FS,
    preprocessing=PREPROCESSING,
    expected_records=EXPECTED_RECORDS,
)

## 1. Hiểu cổng kiểm định trước khi bấm chạy
| Cổng | Điều cần đúng | Khi sai |
|---|---|---|
| Nguồn | Đủ 21.799 dòng; file/metadata đúng SHA256 chính thức | Giải quyết nguồn, không sửa checksum |
| Cohort | Mỗi ecg_id xuất hiện đúng một lần trong metadata/records | Kiểm tra bước tạo manifest/xử lý |
| Tín hiệu | Accepted có shape/dtype/units/finite, không lead hằng, checksum đúng | Truy vết raw và quá trình xuất |
| Nhãn | Nhãn/fold/ID trong records khớp metadata bất biến | Không sửa nhãn bằng tay |
| Split | Không giao bệnh nhân; không waveform trùng hoàn toàn qua tập | Điều tra leakage trước train |
| Chuẩn hóa | Fit train, đúng cohort, class order, mean/std hợp lệ và checksum | Fit lại đúng train trong release phù hợp |
| Tính đầy đủ | Không phải fixture, cả ba tập accepted không rỗng | Chưa công bố ready |

Hash waveform chỉ phát hiện trùng hoàn toàn; không phát hiện mọi dạng bản ghi gần trùng. Nhãn nhóm không thay thế thẩm định chuyên môn.

## 2. Môi trường tạo release khác môi trường nhận dữ liệu
Core hiện gắn fingerprint với Python đầy đủ (gồm patch), NumPy/Pandas/SciPy/WFDB, byte của core.py, cấu hình và nguồn. Đổi Python 3.12.x sang patch khác hoặc sửa core có thể khiến audit_release từ chối dù .npy còn nguyên.

Bảng này giải thích guard, không bypass nó. Với máy nhận chỉ cần đọc dữ liệu, không mặc định chạy lại notebook 00–04. Xem phần bàn giao phía dưới và loader 05; kiểm định nhận dữ liệu độc lập runtime cho toàn release là phần chưa có CLI chuyên biệt.

In [ ]:
import json
import hashlib
import platform
import numpy as np
import pandas as pd
import scipy
import wfdb
import ptbxl_pipeline.core as core_module
from IPython.display import display

identity = json.loads((OUTPUT_DIR / "pipeline.json").read_text(encoding="utf-8"))
current_runtime = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__, "wfdb": wfdb.__version__}
runtime_rows = [{"thanh_phan": name, "producer": version, "hien_tai": current_runtime.get(name), "khop": version == current_runtime.get(name)} for name, version in identity["runtime"].items()]
display(pd.DataFrame(runtime_rows))
core_hash = hashlib.sha256(Path(core_module.__file__).read_bytes()).hexdigest()
print("Byte core.py khớp producer:", core_hash == identity["implementation_sha256"])
print("Cấu hình producer:", identity["config"])
print("Fingerprint:", identity["fingerprint"])

## 3. TOÀN BỘ: audit của môi trường tạo dữ liệu
Cell này đọc nguồn và mọi waveform, đồng thời ghi audit/card/manifests; nó không phải kiểm tra nhận file chỉ đọc. Chạy trong môi trường producer phù hợp khi cần tái kiểm định. Nếu guard mismatch, giữ release cũ; xác định đang muốn nhận dữ liệu hay tái tạo bằng một pipeline mới.

Không sửa ready bằng tay, bỏ assertion hoặc giảm expected_records để qua cổng.

In [ ]:
from IPython.display import JSON, Markdown
from ptbxl_pipeline.core import audit_release, CLASSES

audit = audit_release(config)
assert audit.get("ready") is True and audit.get("checks_passed") is True
assert audit.get("fixture") is False and not audit.get("errors")
display(JSON({key: audit[key] for key in ("ready", "checks_passed", "fixture", "audited_at", "source_records", "accepted_records", "errors")}))
print("Báo cáo:", OUTPUT_DIR / "audit.json")

## 4. Dòng chảy cohort: mọi ECG phải được giải thích
Source = accepted + no_label + rejected. Đây là kiểm tra số lượng; no_label và rejected có ý nghĩa khác nhau. Các con số được đọc từ lần chạy hiện tại, không viết cứng kết quả cũ.

In [ ]:
records = pd.read_csv(OUTPUT_DIR / "records.csv", keep_default_na=False, low_memory=False)
metadata = pd.read_csv(OUTPUT_DIR / "metadata.csv", keep_default_na=False, low_memory=False)
counts = records.status.value_counts().reindex(["accepted", "no_label", "rejected"], fill_value=0)
assert len(records) == len(metadata) == EXPECTED_RECORDS
assert counts.sum() == len(records)
assert int(counts["accepted"]) == audit["accepted_records"]
flow = counts.rename_axis("trang_thai").to_frame("so_ECG")
flow["pct_nguon"] = 100 * flow.so_ECG / len(records)
display(flow)
display(records.groupby(["split", "status", "reason"], dropna=False).size().rename("so_ECG").reset_index())
import matplotlib.pyplot as plt
fig, axis = plt.subplots(figsize=(8, 3))
axis.bar(flow.index, flow.so_ECG)
axis.bar_label(axis.containers[0])
axis.set(ylabel="Số ECG", title=f"Cohort từ {len(records)} ECG nguồn — số nhỏ đọc ở bảng phía trên")
plt.show()

## 5. Việc loại mẫu ảnh hưởng nhãn nào?
Bảng là QA cohort theo nhãn/split, không dùng để lựa chọn phép lọc theo test. Đếm một ECG đa nhãn ở mỗi lớp nó mang; vì vậy tổng số nhãn bị loại có thể lớn hơn số ECG bị loại. No-label không có superclass để cộng vào các lớp.

In [ ]:
label_columns = [f"y_{name}" for name in CLASSES]
label_effect = []
for split in ("train", "validation", "test"):
    source_split = records.loc[records.split.eq(split)]
    kept = source_split.loc[source_split.status.eq("accepted")]
    removed = source_split.loc[~source_split.status.eq("accepted")]
    for name in CLASSES:
        column = f"y_{name}"
        label_effect.append({"split": split, "nhan": name, "truoc_QC": int(source_split[column].sum()), "giu_lai": int(kept[column].sum()), "loai": int(removed[column].sum())})
effects = pd.DataFrame(label_effect)
assert effects.truoc_QC.eq(effects.giu_lai + effects.loai).all()
display(effects)
print("NORM kèm bất thường được giữ:", audit["norm_with_abnormal"])
print("Nhóm waveform trùng hoàn toàn:", len(audit["duplicate_groups"]))
print("Nhóm trùng qua split:", len(audit["cross_split_duplicates"]))

## 6. Manifest, checksum và chuỗi provenance
Metadata giữ nguồn nhãn; records giữ mọi quyết định QC; train/validation/test.csv chỉ chứa accepted. pipeline.json chứa cấu hình và fingerprint; normalization.json giữ train stats; audit.json ghi checksum các artifact và thời điểm kiểm định.

Bảng dưới đối chiếu file nhỏ với checksum trong audit. Nó chưa quét lại từng .npy và không thay thế audit đầy đủ. Hash là bằng chứng toàn vẹn so với bản tham chiếu tin cậy, không phải chữ ký chống sửa đồng thời cả file và manifest.

In [ ]:
def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

artifact_rows = []
for name, expected in audit["artifact_sha256"].items():
    actual = file_sha256(OUTPUT_DIR / name)
    artifact_rows.append({"file": name, "khop_SHA256": actual == expected, "sha256": actual})
assert all(row["khop_SHA256"] for row in artifact_rows)
display(pd.DataFrame(artifact_rows))
summaries = []
for split in ("train", "validation", "test"):
    manifest = pd.read_csv(OUTPUT_DIR / f"{split}.csv", keep_default_na=False, low_memory=False)
    assert manifest.status.eq("accepted").all() and manifest.split.eq(split).all()
    assert manifest.ecg_id.is_unique and len(manifest) == audit["split_counts"][split]
    summaries.append({"split": split, "ECG": len(manifest), "benh_nhan": manifest.patient_id.nunique()})
display(pd.DataFrame(summaries))
display(Markdown((OUTPUT_DIR / "DATASET_CARD.vi.md").read_text(encoding="utf-8")))

## 7. Bàn giao sang máy trường: hai tình huống khác nhau
**Chỉ dùng release đã có:** mang toàn bộ thư mục release (khoảng 5,18 GB ở bản 24/10), source code hiện tại, requirements và các hướng dẫn. Giữ bản tham chiếu hash từ máy gửi; kiểm tra file sau copy trước train. Loader 05 đối chiếu manifest/normalization và từng .npy khi đọc; một batch smoke test chỉ bao phủ những ECG đã đọc. Chưa có CLI độc lập runtime để quét nhận toàn bộ release; không coi smoke test là chứng nhận toàn bộ file đã chuyển.

**Tái tạo hoặc chạy producer audit:** thêm nguồn records500 khoảng 2,65 GB và đúng môi trường producer. Đọc runtime trong pipeline.json, không chỉ dựa dòng “Python 3.12”. Nếu thực sự đổi implementation/runtime, tạo release mới; không xóa release cũ hoặc sửa fingerprint để qua guard.

Dữ liệu và artifacts bị Git ignore. Code/notebook đang ở working tree và chưa được commit: clone remote đơn thuần có thể thiếu phần mới. Chuyển bản làm việc có chọn lọc hoặc đồng bộ Git khi Line cho phép commit/push. Không copy .venv giữa hai hệ điều hành; tạo môi trường riêng từ requirements.

Giữ cùng nhau signals, sidecars, metadata, records, ba manifest, normalization, pipeline, audit và dataset card. Không tách X/Y rồi nối bằng thứ tự suy đoán. Đường dẫn trong release là tương đối; dùng PTBXL_OUTPUT_DIR nếu máy trường có bố trí ổ đĩa khác.

## Quyết định và phạm vi xác nhận
Chỉ sang 05 khi producer audit ready thật, không còn lỗi và file bàn giao đã được kiểm tra phù hợp phạm vi. Hiệu năng model, tính đúng lâm sàng, near-duplicate, và tổng quát hóa ngoài PTB-XL chưa được chứng minh. Nếu một gate fail, giữ dữ liệu cũ và sửa nguyên nhân hoặc tạo release mới. Không lấy việc có notebook/plot làm tiêu chí thay cho bằng chứng.